<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Allison/MachineLearningTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drive.mount('/content/drive')

#Loading Data
path = "/content/drive/MyDrive/sparcs_model_v1.feather"
data = pd.read_feather(path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Looking at the Dataset Info
print(data.shape)
data.info()
data.head()

(4238636, 33)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238636 entries, 0 to 4238635
Data columns (total 33 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   health_service_area      object 
 1   hospital_county          object 
 2   facility_id              object 
 3   age_group                object 
 4   zip_code                 object 
 5   gender                   object 
 6   race                     object 
 7   ethnicity                object 
 8   length_of_stay           int64  
 9   admission_type           object 
 10  disposition              object 
 11  discharge_year           int64  
 12  ccsr_dx_code             object 
 13  ccsr_px_code             object 
 14  apr_drg_code             object 
 15  apr_mdc_code             object 
 16  apr_severity_code        object 
 17  apr_mortality_risk       object 
 18  apr_med_surg_desc        object 
 19  total_charges            float64
 20  payer_medicaid           int64  

,health_service_area,hospital_county,facility_id,age_group,zip_code,gender,race,ethnicity,length_of_stay,admission_type,...,payer_self_pay,payer_blue_cross,payer_other,payer_gov_va,payer_corrections,payer_managed_care,num_payment_types,log_total_charges,los_log,los_yj
0,New York City,Bronx,3058,50-69,104,F,Other Race,Spanish/Hispanic,1,Emergency,...,0,0,0,0,0,0,1,10.217627,0.693147,-1.501001
1,New York City,Bronx,1168,30-49,104,M,Black/African American,Not Span/Hispanic,4,Emergency,...,0,0,0,0,0,0,1,11.322140,1.609438,0.246954
2,New York City,Bronx,3058,50-69,104,M,Other Race,Not Span/Hispanic,4,Emergency,...,0,0,0,0,0,0,2,11.241267,1.609438,0.246954
3,New York City,Bronx,1169,18-29,104,M,Black/African American,Not Span/Hispanic,5,Emergency,...,0,0,0,0,0,0,1,11.258467,1.791759,0.503353
4,New York City,Bronx,1169,50-69,104,F,Other Race,Spanish/Hispanic,3,Emergency,...,0,0,0,0,0,0,2,11.057713,1.386294,-0.103046


In [5]:
#Looking at columns, which data would be useful to be included
print(data.columns.tolist())

['health_service_area', 'hospital_county', 'facility_id', 'age_group', 'zip_code', 'gender', 'race', 'ethnicity', 'length_of_stay', 'admission_type', 'disposition', 'discharge_year', 'ccsr_dx_code', 'ccsr_px_code', 'apr_drg_code', 'apr_mdc_code', 'apr_severity_code', 'apr_mortality_risk', 'apr_med_surg_desc', 'total_charges', 'payer_medicaid', 'payer_medicare', 'payer_private_insurance', 'payer_self_pay', 'payer_blue_cross', 'payer_other', 'payer_gov_va', 'payer_corrections', 'payer_managed_care', 'num_payment_types', 'log_total_charges', 'los_log', 'los_yj']


In [6]:
#Looking at ZIP Codes
print("Unique ZIP codes:", data['zip_code'].nunique())
print(sorted(data['zip_code'].unique()))


Unique ZIP codes: 50
['100', '101', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', 'OOS']


In [7]:
drop_columns=['disposition', 'discharge_year', 'ccsr_px_code', 'apr_med_surg_desc']

In [8]:
#Creating a function to clean and prepare data set
def clean_data(df):

#Remove Duplicates
  df = df.drop_duplicates()

#Excluding the rows that do not have LOS; Missing data
  df = df.dropna(subset = ['length_of_stay'])

#Convert data to numerical, Drop those who can't convert
  df['length_of_stay'] = pd.to_numeric(df['length_of_stay'], errors='coerce')
  df = df.dropna(subset = ['length_of_stay'])

#Remove out-of-state ZIP Codes ('OOS')
  if 'Zip Code' in df.columns:
      df = df[df['zip_code'] != 'OOS']

  df = df.drop(columns =[c for c in drop_columns if c in df.columns], errors = 'ignore')

  return df

#Clean Data
data = clean_data(data)

print("Shape of cleaned data:", data.shape)  # rows, columns




Shape of cleaned data: (4206911, 29)


In [9]:
numerical_columns = ['payer_medicaid', 'payer_medicare','payer_private_insurance','payer_self_pay',
                     'payer_blue_cross','payer_other', 'payer_gov_va',
                     'payer_corrections','payer_managed_care','num_payment_types']


categorical_cols = ['zip_code', 'health_service_area', 'facility_id', 'hospital_county', 'age_group',
                    'gender', 'race', 'ethnicity', 'admission_type', 'apr_mortality_risk',
                    'apr_severity_code', 'apr_drg_code', 'apr_mdc_code', 'ccsr_dx_code']

In [ ]:
#One-hot encoded
data_encoded = pd.get_dummies(data[categorical_cols], drop_first=True)

#Add LOS for correlation
data_encoded['length_of_stay'] = data['length_of_stay']

#Calculate correlations between LOS and all one-hot encoded categorical variables
correlations_cat = data_encoded.corr()['length_of_stay'].sort_values(ascending=False)

print(correlations_cat)

#mean LOS per category, which categories have longer or shorter stays
for col in categorical_cols:
    print(f"\nAverage LOS by {col}:")
    print(data.groupby(col)['length_of_stay'].mean().sort_values(ascending=False).head(10))

In [ ]:
# ZIP code correlations
correlations_zip = correlations_cat[[col for col in correlations_cat.index if col.startswith('zip_code')]]

# Hospital county correlations
correlations_county = correlations_cat[[col for col in correlations_cat.index if col.startswith('hospital_county')]]

#Top 10
top_zipcode_corr = correlations_zip.sort_values(ascending = False).head(10)
top_county_corr = correlations_county.sort_values(ascending = False).head(10)

print("Top ZIP correlations:\n", top_zipcode_corr)
print("\nTop County correlations:\n", top_county_corr)


plt.boxplot([correlations_zip.values, correlations_county.drop.values],
            labels=['ZIP', 'County'])
plt.ylabel("Correlation with LOS")
plt.title("Comparison of ZIP vs County Correlations with LOS")
plt.show()